In [11]:
import sys
!{sys.executable} -m pip install numpy h5py scikit-learn plotly





[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [12]:
import numpy as np
import h5py
import plotly.graph_objects as go
import sys
from pathlib import Path
from plotly.subplots import make_subplots
sys.path.append(str(Path("../scripts").resolve()))
from pca_point_clouds import center_and_scale

print(sys.executable)


/usr/local/bin/python3.12


In [22]:
DATASET_NAME = "airplanes"
CLASS_ID = "02691156"

"choose category of items to analyze"

OUTPUT_DIR = Path(f"../outputs/{DATASET_NAME}/pca_point_clouds")
H5_PATH = Path(f"../data/{CLASS_ID}.h5")


with h5py.File(H5_PATH, "r") as f:
    point_clouds = f["point_clouds"][:].astype(np.float64)

mean_shape = np.load(OUTPUT_DIR / "pca_mean.npy")
components = np.load(OUTPUT_DIR / "eigenvectors.npy")
scores = np.load(OUTPUT_DIR / "pca_scores.npy")
explained = np.load(OUTPUT_DIR / "explained_variance_ratio.npy")
eigenvalues = np.load(OUTPUT_DIR / "explained_variance.npy")
pc1_minus = np.loadtxt(OUTPUT_DIR / "pc1_sigma_-2.0.xyz")
pc1_mean = np.loadtxt(OUTPUT_DIR / "pc1_sigma_+0.0.xyz")
pc1_plus = np.loadtxt(OUTPUT_DIR / "pc1_sigma_+2.0.xyz")
pc2_minus = np.loadtxt(OUTPUT_DIR / "pc2_sigma_-2.0.xyz")
pc2_mean = np.loadtxt(OUTPUT_DIR / "pc2_sigma_+0.0.xyz")
pc2_plus = np.loadtxt(OUTPUT_DIR / "pc2_sigma_+2.0.xyz")

point_clouds_norm = center_and_scale(point_clouds)

print("mean_shape:", mean_shape.shape)
print("components:", components.shape)
print("scores:", scores.shape)
print("explained:", explained.shape)
print("eigenvalues:", eigenvalues.shape)
print(point_clouds.shape)

mean_shape: (2048, 3)
components: (100, 6144)
scores: (100, 100)
explained: (100,)
eigenvalues: (100,)
(100, 2048, 3)


In [23]:
def plot_point_cloud(points, title="Point cloud", size=2):
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=points[:, 0],
        y=points[:, 1],
        z=points[:, 2],
        mode="markers",
        marker=dict(size=size, opacity=0.8),
        name="Point cloud"
    ))


    fig.add_trace(go.Scatter3d(
        x=[0],
        y=[0],
        z=[0],
        mode="markers+text",
        marker=dict(size=3, color="red"),
        text=["Origin"],
        textposition="top center",
        name="Origin"
    ))
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(title="X", showgrid=True, zeroline=True),
            yaxis=dict(title="Y", showgrid=True, zeroline=True),
            zaxis=dict(title="Z", showgrid=True, zeroline=True),
            aspectmode="data"), width=800,
    height=700,
    margin=dict(l=0, r=0, t=40, b=0))

    
    
    

    fig.show()

In [24]:
cumulative = np.cumsum(explained)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=np.arange(1, len(explained) + 1),
    y=explained,
    name="Explained variance"
))

fig.add_trace(go.Scatter(
    x=np.arange(1, len(cumulative) + 1),
    y=cumulative,
    mode="lines+markers",
    name="Cumulative variance",
    yaxis="y2"

))


fig.add_hline(y=0.95, line_dash="dash", annotation_text="95%")

fig.update_layout(
    title="PCA Explained Variance",
    xaxis_title="Principal component",
    yaxis=dict(title="Explained variance ratio"),
    yaxis2=dict(title="", overlaying="y", side="right", range=[0, 1]),
    width=900,
    height=500
)

fig.show()

In [25]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.arange(1, len(eigenvalues) + 1),
    y=eigenvalues,
    mode="lines+markers",
    name="Eigenvalues"
))

fig.update_layout(
    title="PCA Eigenvalues",
    xaxis_title="PC Index",
    yaxis_title="Eigenvalue",
    width=850,
    height=500
)

fig.show()

In [17]:
def plot_pca_space(scores, axis_length=None):
    if axis_length is None:
        axis_length = np.max(np.abs(scores[:, :3]))

    fig = go.Figure()

    
    fig.add_trace(go.Scatter3d(
        x=scores[:, 0],
        y=scores[:, 1],
        z=scores[:, 2],
        mode="markers",
        marker=dict(size=5, opacity=0.75),
        name="Shapes"
    ))

    
    axes = [
        ("PC1", [axis_length, 0, 0], "red"),
        ("PC2", [0, axis_length, 0], "green"),
        ("PC3", [0, 0, axis_length], "blue"),
    ]

    for name, vec, color in axes:
        fig.add_trace(go.Scatter3d(
            x=[0, vec[0]],
            y=[0, vec[1]],
            z=[0, vec[2]],
            mode="lines+markers+text",
            line=dict(color=color, width=8),
            marker=dict(size=4, color=color),
            text=["", name],
            textposition="top center",
            name=name
        ))

    fig.update_layout(
        title="PCA Space with Principal Components",
        scene=dict(
            xaxis_title="PC1",
            yaxis_title="PC2",
            zaxis_title="PC3",
            aspectmode="data"
        ),
        width=850,
        height=750
    )

    fig.show()

plot_pca_space(scores)

In [18]:
def plot_three_point_clouds(points_list, titles, size=2, max_points=None, seed=0):
    fig = make_subplots(
        rows=1,
        cols=3,
        specs=[[{"type": "scene"}, {"type": "scene"}, {"type": "scene"}]],
        subplot_titles=titles
    )

    # stessi indici per tutti i subplot
    sample_idx = None
    if max_points is not None and len(points_list[0]) > max_points:
        rng = np.random.default_rng(seed)
        sample_idx = rng.choice(len(points_list[0]), max_points, replace=False)

    for i, points in enumerate(points_list):
        pts = points

        if sample_idx is not None:
            pts = pts[sample_idx]

        fig.add_trace(
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="markers",
                marker=dict(size=size, opacity=0.8),
                showlegend=False
            ),
            row=1,
            col=i + 1
        )

        fig.add_trace(
            go.Scatter3d(
                x=[0],
                y=[0],
                z=[0],
                mode="markers",
                marker=dict(size=3, color="red"),
                showlegend=False
            ),
            row=1,
            col=i + 1
        )

    for i in range(1, 4):
        fig.update_layout(**{
            f"scene{i}": dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data"
            )
        })

    fig.update_layout(
        width=1200,
        height=420,
        margin=dict(l=0, r=0, t=40, b=0)
    )

    fig.show()

plot_three_point_clouds(
[pc1_minus, pc1_mean, pc1_plus],
["PC1 - 2sigma", "Mean", "PC1 + 2sigma"],
size=2,
max_points=1500
)
plot_three_point_clouds(
[pc2_minus, pc2_mean, pc2_plus],
["PC2 - 2sigma", "Mean", "PC2 + 2sigma"],
size=2,
max_points=1500
)

In [19]:
def reconstruct_from_pca(mean_shape, components, scores, shape_index, k):
    mean_vector = mean_shape.reshape(-1)
    reconstructed = mean_vector + scores[shape_index, :k] @ components[:k, :]
    return reconstructed.reshape(mean_shape.shape)

def plot_original_vs_reconstructions(original, reconstructions, titles, size=2, max_points=1500, seed=0):
    n_plots = 1 + len(reconstructions)

    fig = make_subplots(
        rows=1,
        cols=n_plots,
        specs=[[{"type": "scene"} for _ in range(n_plots)]],
        subplot_titles=["Original"] + titles
    )

    all_shapes = [original] + reconstructions

    sample_idx = None
    if max_points is not None and len(original) > max_points:
        rng = np.random.default_rng(seed)
        sample_idx = rng.choice(len(original), max_points, replace=False)

    for i, points in enumerate(all_shapes):
        pts = points

        if sample_idx is not None:
            pts = pts[sample_idx]

        fig.add_trace(
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="markers",
                marker=dict(size=size, opacity=0.8),
                showlegend=False
            ),
            row=1,
            col=i + 1
        )

        fig.add_trace(
            go.Scatter3d(
                x=[0],
                y=[0],
                z=[0],
                mode="markers",
                marker=dict(size=3, color="red"),
                showlegend=False
            ),
            row=1,
            col=i + 1
        )

    layout_update = {
        "width": 300 * n_plots,
        "height": 420,
        "margin": dict(l=0, r=0, t=40, b=0)
    }

    for i in range(1, n_plots + 1):
        layout_update[f"scene{i}"] = dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data"
        )

    fig.update_layout(**layout_update)
    fig.show()

shape_index = 20 

original = point_clouds_norm[shape_index]

k_comp = [3, 10, 30, 50, 93]

reconstructions = [
    reconstruct_from_pca(mean_shape, components, scores, shape_index, k)
    for k in k_comp
]

titles = [f"k={k}" for k in k_comp]

plot_original_vs_reconstructions(
    original,
    reconstructions,
    titles,
    size=2,
    max_points=1500,
    seed=0
)

In [20]:
def reconstruction_error_vs_k(point_clouds_norm, mean_shape, components, scores, ks):
    X_original = point_clouds_norm.reshape(point_clouds_norm.shape[0], -1)
    mean_vector = mean_shape.reshape(-1)

    errors_mean = []
    errors_std = []

    for k in ks:
        X_rec = mean_vector + scores[:, :k] @ components[:k, :]

        diff = X_original - X_rec
        per_shape_error = np.linalg.norm(diff.reshape(point_clouds_norm.shape), axis=2).mean(axis=1)

        errors_mean.append(per_shape_error.mean())
        errors_std.append(per_shape_error.std())

    return np.array(errors_mean), np.array(errors_std)

ks = [1, 2, 3, 5, 10, 20, 40, 60, 80, 93, 100]
ks = [k for k in ks if k <= components.shape[0]]

errors_mean, errors_std = reconstruction_error_vs_k(point_clouds_norm, mean_shape, components, scores, ks)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ks,
    y=errors_mean,
    mode="lines+markers",
    name="Mean reconstruction error",
    error_y=dict(
        type="data",
        array=errors_std,
        visible=True
    )
))

fig.update_layout(
    title="Reconstruction Error",
    xaxis_title="Number of PCA components (k)",
    yaxis_title="Mean point-wise reconstruction error",
    width=850,
    height=500
)

fig.show()

In [21]:
def generate_random_shape(mean_shape, components, eigenvalues, k, seed=None, strength=1.0):
    rng = np.random.default_rng(seed)

    std = np.sqrt(eigenvalues[:k])
    coeffs = rng.normal(loc=0.0, scale=std * strength, size=k)

    generated = mean_shape.reshape(-1) + coeffs @ components[:k, :]
    return generated.reshape(mean_shape.shape), coeffs
